In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("cleaned1.csv")

In [2]:
df

,customer_id,age,occupation_status,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,delinquencies_last_2yrs,derogatory_marks,product_type,loan_intent,loan_amount,interest_rate,debt_to_income_ratio,loan_to_income_ratio,payment_to_income_ratio,loan_status
0,CUST100000,40,Employed,17.2,25579,692,5.3,895,10820,0,0,0,Credit Card,Business,600,17.02,0.423,0.023,0.008,1.0
1,CUST100001,33,Employed,7.3,43087,627,3.5,169,16550,0,1,0,Personal Loan,Home Improvement,53300,14.10,0.384,1.237,0.412,0.0
2,CUST100002,42,Student,1.1,20840,689,8.4,17,7852,0,0,0,Credit Card,Debt Consolidation,2100,18.33,0.377,0.101,0.034,1.0
3,CUST100003,53,Student,0.5,29147,692,9.8,1480,11603,0,1,0,Credit Card,Business,2900,18.74,0.398,0.099,0.033,1.0
4,CUST100004,32,Employed,12.5,63657,630,7.2,209,12424,0,0,0,Personal Loan,Education,99600,13.92,0.195,1.565,0.522,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45925,CUST145925,28,Employed,6.8,37060,533,4.3,42,9552,0,0,0,Credit Card,Home Improvement,24400,22.25,0.258,0.658,0.219,0.0
45926,CUST145926,43,Employed,3.1,55417,582,18.5,1995,6168,0,4,1,Personal Loan,Home Improvement,66000,15.28,0.111,1.191,0.397,0.0
45927,CUST145927,36,Employed,10.6,69803,632,1.3,667,28097,0,0,0,Credit Card,Debt Consolidation,70000,20.21,0.403,1.003,0.334,0.0
45928,CUST145928,40,Employed,2.4,44935,689,7.6,3529,6324,0,1,0,Personal Loan,Medical,62300,12.62,0.141,1.386,0.462,0.0


## Feature Engineering

We will perform the following feature engineering steps:
1.  **Handle Missing Values**: Address `NaN` values, particularly in the target variable `loan_status` and `payment_to_income_ratio`.
2.  **Categorical Encoding**: Convert categorical features into a numerical format using one-hot encoding.
3.  **Numerical Feature Scaling**: Scale numerical features to a standard range using `StandardScaler`.

### Step 1: Handle Missing Values

First, we will check for missing values across the DataFrame. We'll drop rows where `loan_status` is missing, as it's likely our target variable. For numerical features with missing values, we'll impute them with the mean.

In [3]:
print('Missing values before handling:')
display(df.isnull().sum()[df.isnull().sum() > 0])

# Drop rows where 'loan_status' is NaN (assuming it's the target variable)
df.dropna(subset=['loan_status'], inplace=True)

# Impute missing values in 'payment_to_income_ratio' with the mean
if 'payment_to_income_ratio' in df.columns and df['payment_to_income_ratio'].isnull().any():
    df['payment_to_income_ratio'].fillna(df['payment_to_income_ratio'].mean(), inplace=True)

print('\nMissing values after handling:')
display(df.isnull().sum()[df.isnull().sum() > 0])

Missing values before handling:


,0
debt_to_income_ratio,1
loan_to_income_ratio,1
payment_to_income_ratio,1
loan_status,1



Missing values after handling:


,0


### Step 2: Categorical Encoding

We will identify categorical columns and apply one-hot encoding to convert them into numerical features. The `customer_id` column will be dropped as it's a unique identifier and not useful for modeling.

In [4]:
# Drop 'customer_id' column as it's just an identifier
df = df.drop('customer_id', axis=1)

# Identify categorical columns (object type)
categorical_cols = df.select_dtypes(include='object').columns
print(f"Categorical columns identified: {list(categorical_cols)}")

# Apply one-hot encoding
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print('\nDataFrame after one-hot encoding (first 5 rows):')
display(df.head())

Categorical columns identified: ['occupation_status', 'product_type', 'loan_intent']

DataFrame after one-hot encoding (first 5 rows):


,age,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,delinquencies_last_2yrs,derogatory_marks,loan_amount,interest_rate,debt_to_income_ratio,loan_to_income_ratio,payment_to_income_ratio,loan_status,occupation_status_Self-Employed,occupation_status_Student,product_type_Line of Credit,product_type_Personal Loan,loan_intent_Debt Consolidation,loan_intent_Education,loan_intent_Home Improvement,loan_intent_Medical,loan_intent_Personal
0,40,17.2,25579,692,5.3,895,10820,0,0,0,600,17.02,0.423,0.023,0.008,1.0,False,False,False,False,False,False,False,False,False
1,33,7.3,43087,627,3.5,169,16550,0,1,0,53300,14.10,0.384,1.237,0.412,0.0,False,False,False,True,False,False,True,False,False
2,42,1.1,20840,689,8.4,17,7852,0,0,0,2100,18.33,0.377,0.101,0.034,1.0,False,True,False,False,True,False,False,False,False
3,53,0.5,29147,692,9.8,1480,11603,0,1,0,2900,18.74,0.398,0.099,0.033,1.0,False,True,False,False,False,False,False,False,False
4,32,12.5,63657,630,7.2,209,12424,0,0,0,99600,13.92,0.195,1.565,0.522,1.0,False,False,False,True,False,True,False,False,False


### Step 3: Numerical Feature Scaling

We will scale the numerical features using `StandardScaler` to ensure that no single feature dominates the model due to its magnitude. The `loan_status` column (target) will be excluded from scaling.

In [5]:
# Identify numerical columns to scale (exclude 'loan_status')
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
if 'loan_status' in numerical_cols:
    numerical_cols.remove('loan_status')

print(f"Numerical columns to scale: {list(numerical_cols)}")

# Initialize StandardScaler
scaler = StandardScaler()

# Apply scaling to numerical columns
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

print('\nDataFrame after numerical feature scaling (first 5 rows):')
display(df.head())

Numerical columns to scale: ['age', 'years_employed', 'annual_income', 'credit_score', 'credit_history_years', 'savings_assets', 'current_debt', 'defaults_on_file', 'delinquencies_last_2yrs', 'derogatory_marks', 'loan_amount', 'interest_rate', 'debt_to_income_ratio', 'loan_to_income_ratio', 'payment_to_income_ratio']

DataFrame after numerical feature scaling (first 5 rows):


,age,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,delinquencies_last_2yrs,derogatory_marks,loan_amount,interest_rate,debt_to_income_ratio,loan_to_income_ratio,payment_to_income_ratio,loan_status,occupation_status_Self-Employed,occupation_status_Student,product_type_Line of Credit,product_type_Personal Loan,loan_intent_Debt Consolidation,loan_intent_Education,loan_intent_Home Improvement,loan_intent_Medical,loan_intent_Personal
0,0.452949,1.276672,-0.750583,0.745296,-0.398974,-0.204271,-0.261553,-0.238299,-0.656765,-0.357555,-1.242752,0.374318,0.857686,-1.457077,-1.454853,1.0,False,False,False,False,False,False,False,False,False
1,-0.176814,-0.022389,-0.212973,-0.257702,-0.648797,-0.259540,0.172095,-0.238299,0.525509,-0.357555,0.775743,-0.342865,0.613764,1.144617,1.142457,0.0,False,False,False,True,False,False,True,False,False
2,0.632881,-0.835943,-0.896101,0.699004,0.031277,-0.271112,-0.486173,-0.238299,-0.656765,-0.357555,-1.185300,0.696067,0.569983,-1.289917,-1.287700,1.0,False,True,False,False,True,False,False,False,False
3,1.622507,-0.914674,-0.641022,0.745296,0.225584,-0.159737,-0.202296,-0.238299,0.525509,-0.357555,-1.154658,0.796767,0.701326,-1.294203,-1.294129,1.0,False,True,False,False,False,False,False,False,False
4,-0.266780,0.659946,0.418660,-0.211409,-0.135271,-0.256495,-0.140162,-0.238299,-0.656765,-0.357555,2.549107,-0.387075,-0.568321,1.847546,1.849646,1.0,False,False,False,True,False,True,False,False,False


### Step 4: Numerical Feature Scaling with MinMaxScaler

Now, we will also apply `MinMaxScaler` to scale the numerical features to a range between 0 and 1. Note that this will be applied on the data that has already been scaled by `StandardScaler` in the previous step.

In [6]:
# Initialize MinMaxScaler
minmax_scaler = MinMaxScaler()

# Apply scaling to numerical columns
df[numerical_cols] = minmax_scaler.fit_transform(df[numerical_cols])

print('\nDataFrame after numerical feature scaling with MinMaxScaler (first 5 rows):')
display(df.head())


DataFrame after numerical feature scaling with MinMaxScaler (first 5 rows):


,age,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,delinquencies_last_2yrs,derogatory_marks,loan_amount,interest_rate,debt_to_income_ratio,loan_to_income_ratio,payment_to_income_ratio,loan_status,occupation_status_Self-Employed,occupation_status_Student,product_type_Line of Credit,product_type_Personal Loan,loan_intent_Debt Consolidation,loan_intent_Education,loan_intent_Home Improvement,loan_intent_Medical,loan_intent_Personal
0,0.423077,0.431078,0.045017,0.685259,0.176667,0.002983,0.065897,0.0,0.000000,0.0,0.001005,0.648235,0.527569,0.007526,0.007530,1.0,False,False,False,False,False,False,False,False,False
1,0.288462,0.182957,0.119519,0.555777,0.116667,0.000563,0.100990,0.0,0.111111,0.0,0.530653,0.476471,0.478697,0.616658,0.615964,0.0,False,False,False,True,False,False,True,False,False
2,0.461538,0.027569,0.024851,0.679283,0.280000,0.000057,0.047721,0.0,0.000000,0.0,0.016080,0.725294,0.469925,0.046663,0.046687,1.0,False,True,False,False,True,False,False,False,False
3,0.673077,0.012531,0.060200,0.685259,0.326667,0.004933,0.070693,0.0,0.111111,0.0,0.024121,0.749412,0.496241,0.045660,0.045181,1.0,False,True,False,False,False,False,False,False,False
4,0.269231,0.313283,0.207051,0.561753,0.240000,0.000697,0.075721,0.0,0.000000,0.0,0.995980,0.465882,0.241855,0.781234,0.781627,1.0,False,False,False,True,False,True,False,False,False


In [7]:
df.to_csv("featured.csv",index=False)